In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# 🏥 Clinically-Weighted Loss for High-Risk Retinal Lesions

When detecting Diabetic Retinopathy (DR), not all lesions are created equal. While missing a microaneurysm (MA) might not be immediately life-altering, missing severe pathologies like Neovascularization (NV) or Retinal Detachment (RD) can lead to irreversible blindness. 

**The Problem:** Traditional machine learning approaches usually balance datasets by looking at *how rare* a class is (inverse class frequency). But what if a lesion is both rare AND highly dangerous? Relying solely on statistical rarity ignores the **clinical reality** of the disease.

**The Solution:** We are implementing a **Hybrid Clinically-Weighted Loss** for our `MultiScaleLesionAttentionNet` architecture. This approach:
1. Calculates standard inverse-frequency weights to handle natural class imbalance.
2. Multiplies those statistical weights by a hand-crafted **Clinical Risk Multiplier**, created with input from ophthalmologists.
3. Normalizes the final weights so the math stays stable during training.

In [14]:
import os
import ast
import argparse
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from sklearn.model_selection import train_test_split
from tqdm import tqdm

In [4]:
import torch
import torch.nn as nn

def compute_hybrid_pos_weights(class_frequencies, device='cuda'):
    """
    Computes a hybrid weight tensor that combines statistical rarity (inverse frequency)
    with clinical risk severity for 7 retinal lesion classes.
    """
    frequencies = torch.tensor(class_frequencies, dtype=torch.float32, device=device)
    
    # 1. Standard Inverse Frequency Weights
    epsilon = 1e-6
    inverse_freq_weights = 1.0 / (frequencies + epsilon)
    
    # 2. Clinical Risk Multipliers
    # [MA=1, HE=1, IH=2, CWS=2, NV=4, VH=4, RD=5]
    clinical_risk_multipliers = torch.tensor(
        [1.0, 1.0, 2.0, 2.0, 4.0, 4.0, 5.0], 
        dtype=torch.float32, 
        device=device
    )
    
    # 3. Hybrid Weights
    hybrid_weights = inverse_freq_weights * clinical_risk_multipliers
    
    # 4. Normalize (mean = 1.0)
    mean_weight = hybrid_weights.mean()
    normalized_hybrid_weights = hybrid_weights / mean_weight
    
    return normalized_hybrid_weights

# Setup device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}\n")

# EXACT frequencies mined from MMRDR/MMRDR-UWF/UWF.csv
# MA: 6938, HE: 3846, IH: 3849, CWS: 1010, NV: 1188, VH: 1219, RD: 238
actual_frequencies = [6938, 3846, 3849, 1010, 1188, 1219, 238]

pos_weights = compute_hybrid_pos_weights(actual_frequencies, device=device)

print("Final Normalized Hybrid Weights (pos_weight):")
classes = ["MA", "HE", "IH", "CWS", "NV", "VH", "RD"]
for cls_name, weight in zip(classes, pos_weights):
    print(f"{cls_name}: {weight:.4f}")

# Instantiate the Loss Function
lesion_criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weights)

Using device: cuda

Final Normalized Hybrid Weights (pos_weight):
MA: 0.0330
HE: 0.0596
IH: 0.1190
CWS: 0.4536
NV: 0.7712
VH: 0.7516
RD: 4.8120


## 🔄 1. Preparing the Data Pipeline

We load our Ultra-Widefield (UWF) dataset. The pipeline includes:
- Elliptical masking to remove artifacts on UWF boundaries.
- String parsing for multi-label targets.
- Exact transformations requested (Resize to 512, Horizontal/Vertical Flip, Rotation, ColorJitter, Normalization).

In [6]:
import os
import cv2
import numpy as np
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.model_selection import train_test_split

def apply_uwf_mask(pil_img, tightness=0.98):
    """Applies an elliptical mask tailored for UWF images"""
    img = np.array(pil_img)
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    _, thresh = cv2.threshold(gray, 15, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours: return pil_img
    largest_contour = max(contours, key=cv2.contourArea)
    if len(largest_contour) >= 5:
        ellipse = cv2.fitEllipse(largest_contour)
        scaled_ellipse = (ellipse[0], (ellipse[1][0]*tightness, ellipse[1][1]*tightness), ellipse[2])
        mask = np.zeros_like(gray)
        cv2.ellipse(mask, scaled_ellipse, 255, -1)
        clean_img = cv2.bitwise_and(img, img, mask=mask)
        return Image.fromarray(clean_img)
    return pil_img

def resolve_image_path(img_root, image_field):
    image_field = str(image_field).strip().replace("\\", "/")
    image_field = image_field.lstrip("./")
    candidates = [
        os.path.join(img_root, image_field),
        os.path.join(img_root, os.path.basename(image_field)),
        os.path.join(img_root, "img", os.path.basename(image_field)),
        os.path.join(img_root, "img", image_field),
    ]
    for path in set(os.path.normpath(c) for c in candidates):
        if os.path.exists(path): return path
    return None

class MMRDRDataset(Dataset):
    def __init__(self, dataframe, img_dir, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = resolve_image_path(self.img_dir, str(row['image']))
        if img_path is None or not os.path.exists(img_path):
            img_path = os.path.join(self.img_dir, str(row['image']))
            
        try:
            image = Image.open(img_path).convert('RGB')
        except:
            image = Image.new('RGB', (1024, 1024), (0, 0, 0))
            
        image = apply_uwf_mask(image) 
        if self.transform: image = self.transform(image)
            
        grade_label = int(row['grade'])
        lesion_str = str(row['lesion']).strip()
        if lesion_str.startswith('[') and lesion_str.endswith(']'): lesion_str = lesion_str[1:-1]
        try:
            lesion_labels = np.array([float(x.strip()) for x in lesion_str.split(',') if x.strip() != ''], dtype=np.float32)
            if len(lesion_labels) != 7: raise ValueError
        except ValueError:
            lesion_labels = np.zeros(7, dtype=np.float32)
                
        return image, grade_label, lesion_labels

# Transforms matching UWF_custom_model
img_size = 512
train_transform = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load actual CSV dataset
csv_path = '/kaggle/input/datasets/krrish1008/mmrdr-dataset/MMRDR/MMRDR-UWF/UWF.csv'
img_root = '/kaggle/input/datasets/krrish1008/mmrdr-dataset/MMRDR/MMRDR-UWF/img'
df = pd.read_csv(csv_path)
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

train_dataset = MMRDRDataset(train_df, img_root, train_transform)
test_dataset = MMRDRDataset(test_df, img_root, val_transform)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=4, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False, num_workers=4, pin_memory=True)

print(f"DataLoaders prepared! Train Size: {len(train_dataset)}, Test Size: {len(test_dataset)}")

DataLoaders prepared! Train Size: 8323, Test Size: 2081


## 🧠 2. Initializing the Model & Optimizer

We load `MultiScaleLesionAttentionNet` directly from `alternate_architechture.py`.

In [9]:
# 1. THE LOSS PARADIGM: CLASS-BALANCED FOCAL LOSS

class ClassBalancedFocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0):
        super(ClassBalancedFocalLoss, self).__init__()
        self.alpha = alpha  # Expected to be a tensor of normalized weights per class
        self.gamma = gamma

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none', weight=self.alpha)
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        return focal_loss.mean()



In [10]:
# 2. THE CUSTOM ARCHITECTURE: MULTI-SCALE LESION-ATTENTION NET (MS-LAN)
# ==========================================
class MultiScaleLesionAttentionNet(nn.Module):
    def __init__(self, num_grades=5, num_lesions=7):
        super(MultiScaleLesionAttentionNet, self).__init__()
        resnet = models.resnet50(weights='IMAGENET1K_V2')
        
        # Branch 1: Micro details preserved at 256x256 resolution
        self.early_features = nn.Sequential(
            resnet.conv1,    # Stride 2 downsamples 512x512 -> 256x256
            resnet.bn1,
            resnet.relu      # No maxpool here! Preserves microscopic microaneurysms
        ) # Output size: [Batch, 64, 256, 256]
        
        # Branch 2: Deep global context downsampled to 16x16 resolution
        self.deep_features = nn.Sequential(
            resnet.maxpool,  # Downsamples 256x256 -> 128x128
            resnet.layer1,   # [Batch, 256, 128, 128]
            resnet.layer2,   # [Batch, 512, 64, 64]
            resnet.layer3,   # [Batch, 1024, 32, 32]
            resnet.layer4    # [Batch, 2048, 16, 16]
        )
        
        # Multi-Scale Alignment Layers
        self.early_compress = nn.Sequential(
            nn.AdaptiveAvgPool2d((16, 16)), # Shrinks grid layout down to 16x16
            nn.Conv2d(64, 512, kernel_size=1) # Expands channels from 64 to 512
        )
        self.deep_compress = nn.Conv2d(2048, 512, kernel_size=1) # Compresses 2048 to 512
        
        # Head A: Multi-label Lesion Estimation Mask (BCE Tracking)
        self.lesion_head = nn.Sequential(
            nn.Conv2d(512, num_lesions, kernel_size=1),
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten()
        )
        
        # Head B: DR Severity Grading Head with regularization against overfitting
        self.grade_head = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.5), # Regularization barrier
            nn.Linear(256, num_grades)
        )

    def forward(self, x):
        # 1. Run the Micro detail extraction layer
        early_maps = self.early_features(x) # [B, 64, 256, 256]
        
        # 2. Complete the normal deep macro path
        deep_maps = self.deep_features(early_maps) # [B, 2048, 16, 16]
        
        # 3. Shape alignment and channel compression
        f1 = self.early_compress(early_maps) # [B, 512, 16, 16]
        f2 = self.deep_compress(deep_maps)   # [B, 512, 16, 16]
        
        # 4. Feature Fusion (Combines Micro + Macro vectors)
        fused_features = F.relu(f1 + f2) # [B, 512, 16, 16]
        
        # 5. Route features to target specific heads
        lesion_out = self.lesion_head(fused_features) # [B, num_lesions]
        grade_out = self.grade_head(fused_features)   # [B, num_grades]
        
        return grade_out, lesion_out



In [11]:
# 3. PRODUCTION DATA PIPELINE
# ==========================================
class MMRDRMultiTaskDataset(Dataset):
    def __init__(self, dataframe, img_dir, transform=None):
        self.img_dir = img_dir
        self.transform = transform
        self.images = dataframe['image'].values
        self.grades = dataframe['grade'].values
        
        # Parse text lists into numpy matrix to maintain multi-worker tracking safely
        raw_lesions = [ast.literal_eval(x) for x in dataframe['lesion']]
        self.lesions = np.array(raw_lesions, dtype=np.float32)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, self.images[idx])
        image = Image.open(img_path).convert('RGB')
        grade = int(self.grades[idx])
        lesion_tensor = torch.from_numpy(self.lesions[idx])
        
        if self.transform:
            image = self.transform(image)
        return image, grade, lesion_tensor

def execute_mmrdr_split(base_path, modality):
    csv_name = 'FP.csv' if modality.lower() == 'cfp' else f'{modality.upper()}.csv'
    file_path = os.path.join(base_path, f'MMRDR-{modality.upper()}', csv_name)
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Missing resource path error: {file_path}")
    df = pd.read_csv(file_path)
    train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
    return train_df, test_df




In [17]:
import sys
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
sys.path.append('..') # Access root scripts

model = MultiScaleLesionAttentionNet(num_grades=5, num_lesions=7).to(device)
print("Successfully loaded MultiScaleLesionAttentionNet!")

optimizer = optim.Adam(model.parameters(), lr=1e-4)
grade_criterion = nn.CrossEntropyLoss()

MAX_EPOCHS = 50
scheduler = CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS, eta_min=1e-6)
best_val_loss = float('inf')
patience = 8
patience_counter = 0

Successfully loaded MultiScaleLesionAttentionNet!


## 🚀 3. The Training Loop (with Clinical Weights)

This true PyTorch training loop handles forward passes from your actual dataloader. The hybrid lesion criterion severely penalizes missing highly-weighted lesions.## 🚀 3. The Training Loop (with Clinical Weights)

This true PyTorch training loop handles forward passes from your actual dataloader. The hybrid lesion criterion severely penalizes missing highly-weighted lesions.

In [18]:
from tqdm import tqdm
import torch

def train_model(model, optimizer, scheduler, lesion_criterion, grade_criterion, train_loader, val_loader, max_epochs=50):
    # Using python's global keyword to update the tracking state accurately
    global best_val_loss, patience_counter
    print("Starting Training Phase...")
    
    for epoch in range(max_epochs):
        # ==================== TRAINING PHASE ====================
        model.train()
        running_train_loss = 0.0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{max_epochs} [Train]")
        
        for images, grades, lesions in pbar:
            images = images.to(device)
            grades = grades.to(device, dtype=torch.long)
            lesions = lesions.to(device, dtype=torch.float32)
            
            # Forward Pass
            grade_logits, lesion_logits = model(images)
            
            # Calculate Losses
            loss_lesion = lesion_criterion(lesion_logits, lesions)
            loss_grade = grade_criterion(grade_logits, grades)
            total_loss = loss_lesion + loss_grade
            
            # Backward Pass & Optimization
            optimizer.zero_grad()
            total_loss.backward()
            optimizer.step()
            
            running_train_loss += total_loss.item()
            pbar.set_postfix({'loss': f"{total_loss.item():.4f}"})
            
        # Step the learning rate scheduler after the epoch
        scheduler.step()
        avg_train_loss = running_train_loss / len(train_loader)

        # ==================== VALIDATION PHASE ====================
        model.eval()
        running_val_loss = 0.0
        
        with torch.no_grad():
            for images, grades, lesions in val_loader:
                images = images.to(device)
                grades = grades.to(device, dtype=torch.long)
                lesions = lesions.to(device, dtype=torch.float32)
                
                grade_logits, lesion_logits = model(images)
                
                loss_lesion = lesion_criterion(lesion_logits, lesions)
                loss_grade = grade_criterion(grade_logits, grades)
                running_val_loss += (loss_lesion + loss_grade).item()
                
        avg_val_loss = running_val_loss / len(val_loader)
        print(f"\n📢 Epoch [{epoch+1}/{max_epochs}] Finished -> Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")
        print(f"📉 Current Learning Rate: {scheduler.get_last_lr()[0]:.2e}")
        
        # ==================== EARLY STOPPING & CHECKPOINTING ====================
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            patience_counter = 0
            
            # Explicit Kaggle output directory paths
            checkpoint_path = "/kaggle/working/best_sota_model.pth"
            torch.save(model.state_dict(), checkpoint_path)
            print(f"🏆 New best validation loss! Saved checkpoint to {checkpoint_path}\n")
        else:
            patience_counter += 1
            print(f"⚠️ No improvement in validation loss. Early stopping patience: {patience_counter}/{patience}\n")
            
        if patience_counter >= patience:
            print(f"🛑 Early stopping triggered at epoch {epoch+1}. Model converged successfully!")
            break

# Execute the dynamic SOTA training run
train_model(
    model=model, 
    optimizer=optimizer, 
    scheduler=scheduler, 
    lesion_criterion=lesion_criterion, 
    grade_criterion=grade_criterion, 
    train_loader=train_loader, 
    val_loader=test_loader, 
    max_epochs=MAX_EPOCHS
)

Starting Training Phase...


Epoch 1/50 [Train]: 100%|██████████| 1041/1041 [17:26<00:00,  1.01s/it, loss=0.5889]



📢 Epoch [1/50] Finished -> Train Loss: 1.2178 | Val Loss: 0.8357
📉 Current Learning Rate: 9.99e-05
🏆 New best validation loss! Saved checkpoint to /kaggle/working/best_sota_model.pth



Epoch 2/50 [Train]: 100%|██████████| 1041/1041 [17:27<00:00,  1.01s/it, loss=0.7096]



📢 Epoch [2/50] Finished -> Train Loss: 0.9197 | Val Loss: 0.7932
📉 Current Learning Rate: 9.96e-05
🏆 New best validation loss! Saved checkpoint to /kaggle/working/best_sota_model.pth



...


📢 Epoch [20/50] Finished -> Train Loss: 0.5028 | Val Loss: 0.7401
📉 Current Learning Rate: 6.58e-05
⚠️ No improvement in validation loss. Early stopping patience: 6/8



Epoch 21/50 [Train]: 100%|██████████| 1041/1041 [18:00<00:00,  1.04s/it, loss=0.2369]



📢 Epoch [21/50] Finished -> Train Loss: 0.4793 | Val Loss: 0.7144
📉 Current Learning Rate: 6.28e-05
⚠️ No improvement in validation loss. Early stopping patience: 7/8



Epoch 22/50 [Train]: 100%|██████████| 1041/1041 [17:47<00:00,  1.03s/it, loss=0.9749]



📢 Epoch [22/50] Finished -> Train Loss: 0.4609 | Val Loss: 0.7904
📉 Current Learning Rate: 5.98e-05
⚠️ No improvement in validation loss. Early stopping patience: 8/8

🛑 Early stopping triggered at epoch 22. Model converged successfully!


In [19]:
from sklearn.metrics import classification_report, accuracy_score

def evaluate_sota_performance(model, test_loader, checkpoint_path="/kaggle/working/best_sota_model.pth"):
    # Load the absolute best saved weights
    model.load_state_dict(torch.load(checkpoint_path))
    model.eval()
    
    all_grade_preds = []
    all_grade_targets = []
    all_lesion_preds = []
    all_lesion_targets = []
    
    print("Evaluating Best Model on Test Set...")
    with torch.no_grad():
        for images, grades, lesions in tqdm(test_loader, desc="Evaluating"):
            images = images.to(device)
            grade_logits, lesion_logits = model(images)
            
            # Multi-class Grade Predictions
            grade_preds = torch.argmax(grade_logits, dim=1)
            all_grade_preds.extend(grade_preds.cpu().numpy())
            all_grade_targets.extend(grades.numpy())
            
            # Multi-label Lesion Predictions (Threshold = 0.5)
            lesion_preds = torch.sigmoid(lesion_logits) > 0.5
            all_lesion_preds.append(lesion_preds.cpu().numpy())
            all_lesion_targets.append(lesions.numpy())
            
    all_lesion_preds = np.vstack(all_lesion_preds)
    all_lesion_targets = np.vstack(all_lesion_targets)
    
    # 1. DR Grading Accuracy & Report
    grade_acc = accuracy_score(all_grade_targets, all_grade_preds)
    print(f"\n🎯 FINAL DR GRADING ACCURACY: {grade_acc * 100:.2f}%")
    print("\n--- DR GRADING CLASSIFICATION REPORT ---")
    print(classification_report(all_grade_targets, all_grade_preds, zero_division=0))
    
    # 2. Multi-Label Lesion Report (SOTA Medical Standards)
    print("\n--- CLINICAL LESION EVALUATION REPORT ---")
    classes = ["MA", "HE", "IH", "CWS", "NV", "VH", "RD"]
    print(classification_report(all_lesion_targets, all_lesion_preds, target_names=classes, zero_division=0))

# Execute the final performance audit
evaluate_sota_performance(model, test_loader)

Evaluating Best Model on Test Set...


Evaluating: 100%|██████████| 261/261 [04:08<00:00,  1.05it/s]


🎯 FINAL DR GRADING ACCURACY: 76.50%

--- DR GRADING CLASSIFICATION REPORT ---
              precision    recall  f1-score   support

           0       0.80      0.94      0.86       676
           1       0.64      0.53      0.58       399
           2       0.68      0.75      0.71       444
           3       0.78      0.60      0.68       269
           4       0.96      0.86      0.90       293

    accuracy                           0.77      2081
   macro avg       0.77      0.74      0.75      2081
weighted avg       0.76      0.77      0.76      2081


--- CLINICAL LESION EVALUATION REPORT ---
              precision    recall  f1-score   support

          MA       1.00      0.72      0.83      1388
          HE       1.00      0.10      0.19       780
          IH       0.94      0.36      0.52       741
         CWS       0.50      0.04      0.07       193
          NV       0.88      0.68      0.76       238
          VH       0.98      0.77      0.86       226
          